# Notebook 02v2: Collect Hidden States (Bug-Fixed)

Changes from v1:
- `enable_thinking=False` unified across all files
- `num_train=500, num_eval=500` (paper scale)
- Validates Single Answer MATH accuracy as sanity check

In [2]:
import os, sys, subprocess
REPO = 'thoughtcomm'
if os.path.exists(REPO):
    os.chdir(REPO)
    subprocess.run(['git', 'pull', 'origin', 'main'], check=True)
else:
    subprocess.run(['git', 'clone', 'https://github.com/AUMEZAK/thoughtcomm.git'], check=True)
    os.chdir(REPO)
    subprocess.run(['pip', 'install', '-e', '.', '-q'], check=True)
# Verify v2 fixes
# Original line causing error: exec(open('tests/test_v2_fixes.py').read())

# Fix: Run the test script as a subprocess to correctly handle __file__
try:
    subprocess.run([sys.executable, 'tests/test_v2_fixes.py'], check=True)
    print("v2 fixes verification script executed successfully.")
except subprocess.CalledProcessError as e:
    print(f"Error during v2 fixes verification: {e}")
    if e.stdout:
        print("Stdout:", e.stdout.decode())
    if e.stderr:
        print("Stderr:", e.stderr.decode())

v2 fixes verification script executed successfully.


In [3]:
# Mount Google Drive for checkpoints
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import torch
from configs.config import ThoughtCommConfig
from models.model_utils import load_model_and_tokenizer
from data.math_data import load_math_dataset
from data.gsm8k_data import load_gsm8k_dataset
from pipeline.debate import MultiAgentDebate
from pipeline.hidden_state_collector import HiddenStateCollector
import os

config = ThoughtCommConfig.for_qwen_0_6b()
print(f'Model: {config.model_name}')
print(f'num_train={config.num_train}, num_eval={config.num_eval}')
print(f'n_h={config.n_h} ({config.num_agents} agents x {config.hidden_size})')

MODEL_TAG = config.model_name.replace('/', '_')
SAVE_DIR = config.save_dir
os.makedirs(SAVE_DIR, exist_ok=True)

Model: Qwen/Qwen3-0.6B
num_train=500, num_eval=500
n_h=3072 (3 agents x 1024)


In [7]:
# Load model
# Convert torch.dtype object to its string name (e.g., 'torch.bfloat16' -> 'bfloat16')
dtype_str = str(config.torch_dtype).split('.')[-1]
model, tokenizer = load_model_and_tokenizer(
    config.model_name, dtype=dtype_str
)
print(f'Model loaded: {config.model_name}')
print(f'Pad token: {tokenizer.pad_token} (id={tokenizer.pad_token_id})')

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/1.50G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Model loaded: Qwen/Qwen3-0.6B
Pad token: <|endoftext|> (id=151643)


In [8]:
# Load datasets (paper scale: 500+500)
math_train, math_eval = load_math_dataset(
    num_train=config.num_train, num_eval=config.num_eval, level=config.math_level
)
gsm8k_train, gsm8k_eval = load_gsm8k_dataset(
    num_train=config.num_train, num_eval=config.num_eval
)
print(f'MATH: {len(math_train)} train, {len(math_eval)} eval')
print(f'GSM8K: {len(gsm8k_train)} train, {len(gsm8k_eval)} eval')

README.md: 0.00B [00:00, ?B/s]

algebra/train-00000-of-00001.parquet:   0%|          | 0.00/505k [00:00<?, ?B/s]

algebra/test-00000-of-00001.parquet:   0%|          | 0.00/353k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1744 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1187 [00:00<?, ? examples/s]

...

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

MATH: 500 train, 500 eval
GSM8K: 500 train, 500 eval


In [9]:
# Sanity check: Single Answer on 10 MATH problems
# This verifies enable_thinking=False is working
from pipeline.thought_comm import run_single_answer_baseline
from evaluation.math_eval import extract_boxed_answer, grade_answer

sa_sample = run_single_answer_baseline(model, tokenizer, math_eval[:10], config)
correct = 0
for resp, ex in zip(sa_sample, math_eval[:10]):
    pred = extract_boxed_answer(resp)
    if pred and grade_answer(pred, ex['answer']):
        correct += 1
print(f'Single Answer sanity check: {correct}/10 correct ({correct*10}%)')
print('Expected ~40-50% if enable_thinking fix is working')
print('If ~0-5%, enable_thinking fix is NOT applied')

# Show a sample response to verify no <think> tags
print(f'\nSample response (first 200 chars):\n{sa_sample[0][:200]}')

Single Answer Baseline: 100%|██████████| 10/10 [02:26<00:00, 14.66s/it]


Single Answer sanity check: 3/10 correct (30%)
Expected ~40-50% if enable_thinking fix is working
If ~0-5%, enable_thinking fix is NOT applied

Sample response (first 200 chars):
To solve the problem, we are given a sequence:

$$
1,000,000; \ 500,000; \ 250,000; \ \ldots
$$

This sequence is constructed by **repeatedly dividing by 2** starting from 1,000,000.

We can represent


In [14]:
# Collect hidden states for MATH training set
# Fixed: removed tokenizer from HiddenStateCollector arguments to match its __init__ signature
collector = HiddenStateCollector(model, config)
H_math, meta_math = collector.collect(
    math_train, save_dir=os.path.join(SAVE_DIR, f'{MODEL_TAG}_math_v2')
)
print(f'MATH hidden states: {H_math.shape}')

Error on example 0: 'Qwen3ForCausalLM' object has no attribute 'run_debate'. Skipping.
Error on example 1: 'Qwen3ForCausalLM' object has no attribute 'run_debate'. Skipping.
Error on example 2: 'Qwen3ForCausalLM' object has no attribute 'run_debate'. Skipping.
Error on example 3: 'Qwen3ForCausalLM' object has no attribute 'run_debate'. Skipping.
Error on example 4: 'Qwen3ForCausalLM' object has no attribute 'run_debate'. Skipping.
Error on example 5: 'Qwen3ForCausalLM' object has no attribute 'run_debate'. Skipping.
Error on example 6: 'Qwen3ForCausalLM' object has no attribute 'run_debate'. Skipping.
Error on example 7: 'Qwen3ForCausalLM' object has no attribute 'run_debate'. Skipping.
Error on example 8: 'Qwen3ForCausalLM' object has no attribute 'run_debate'. Skipping.
Error on example 9: 'Qwen3ForCausalLM' object has no attribute 'run_debate'. Skipping.
Error on example 10: 'Qwen3ForCausalLM' object has no attribute 'run_debate'. Skipping.
Error on example 11: 'Qwen3ForCausalLM' ob

RuntimeError: No hidden states collected — all examples failed or dataset is empty.

In [ ]:
# Collect hidden states for GSM8K training set
H_gsm8k, meta_gsm8k = collector.collect(
    gsm8k_train, save_dir=os.path.join(SAVE_DIR, f'{MODEL_TAG}_gsm8k_v2')
)
print(f'GSM8K hidden states: {H_gsm8k.shape}')

In [ ]:
# Save combined hidden states
import json

torch.save({'H': H_math, 'metadata': meta_math},
           os.path.join(SAVE_DIR, f'{MODEL_TAG}_math_hidden_v2.pt'))
torch.save({'H': H_gsm8k, 'metadata': meta_gsm8k},
           os.path.join(SAVE_DIR, f'{MODEL_TAG}_gsm8k_hidden_v2.pt'))

# Verify
summary = {
    'model': config.model_name,
    'math_h_shape': list(H_math.shape),
    'math_h_mean': H_math.float().mean().item(),
    'math_h_std': H_math.float().std().item(),
    'math_num_samples': len(meta_math),
    'gsm8k_h_shape': list(H_gsm8k.shape),
    'gsm8k_h_mean': H_gsm8k.float().mean().item(),
    'gsm8k_h_std': H_gsm8k.float().std().item(),
    'gsm8k_num_samples': len(meta_gsm8k),
}

os.makedirs('results', exist_ok=True)
with open(f'results/02v2_hidden_states_{MODEL_TAG}.json', 'w') as f:
    json.dump(summary, f, indent=2)

for k, v in summary.items():
    print(f'{k}: {v}')